In [34]:
## 파일 업로드
import sqlite3
import pandas as pd

from google.colab import files
uploaded = files.upload()


Saving 종목별 재주재표 분석.db to 종목별 재주재표 분석.db


In [35]:
## 컬럼 정상 작동 확
db_path = "종목별 재주재표 분석.db"
conn = sqlite3.connect(db_path)

df = pd.read_sql("SELECT * FROM financials", conn)
df.head()


,stock_code,year,roa,roe,net_margin,sales_growth_rate,asset_growth,op_income_growth,net_income_growth,op_margin,...,asset_turnover_ratio,total_assets,current_assets,non_current_assets,current_liabilities,non_current_liabilities,total_liabilities,capital,equity,retained_earnings
0,372800,2021,-0.053106,-0.903485,-0.038996,NaN,NaN,NaN,NaN,-0.037706,...,1.361833,4.914109e+10,4.230158e+10,6.839509e+09,1.419986e+10,4.844733e+09,1.904459e+10,2.888490e+09,2.888490e+09,3.647553e+09
1,372800,2022,-0.240173,-3.348948,-0.183398,-0.184550,-0.152006,3.492101,2.835034,-0.207711,...,1.309569,4.167134e+10,3.058691e+10,1.108443e+10,1.601802e+10,5.211660e+08,1.653918e+10,2.988495e+09,2.988495e+09,-6.312810e+09
2,372800,2023,-0.147384,-2.355391,-0.094704,0.373413,0.155704,-0.436433,-0.290794,-0.085232,...,1.556263,4.815975e+10,3.663765e+10,1.152210e+10,3.054788e+10,3.702691e+08,3.091815e+10,3.013495e+09,3.013495e+09,-1.306519e+10
3,372800,2024,0.020907,0.411933,0.015599,0.061760,0.232904,-1.026072,-1.174889,0.002093,...,1.340233,5.937634e+10,4.604766e+10,1.332867e+10,4.099755e+10,2.202691e+08,4.121782e+10,3.013495e+09,3.013495e+09,-1.163791e+10
4,372800,2025,-0.070209,-1.090559,-0.160892,-0.742468,-0.209044,0.885324,-3.656205,0.015322,...,0.436375,4.696405e+10,3.026478e+10,1.669927e+10,3.452251e+10,2.458093e+08,3.476832e+10,3.023495e+09,3.023495e+09,-1.743624e+10


In [36]:
# 민규님이 제안하신 내용 추가

df["debt_to_assets"] = df["total_liabilities"] / df["total_assets"]

df["non_current_debt_ratio"] = df["non_current_liabilities"] / df["total_assets"]
df["current_debt_ratio"] = df["current_liabilities"] / df["total_assets"]

df["equity_multiplier"] = df["total_assets"] / df["equity"].replace(0, pd.NA)
df["short_term_debt_ratio"] = (
    df["current_liabilities"] / df["total_liabilities"].replace(0, pd.NA)
)

df["current_assets_ratio"] = df["current_assets"] / df["total_assets"]
df["non_current_assets_ratio"] = df["non_current_assets"] / df["total_assets"]

df["net_working_capital_ratio"] = (
    (df["current_assets"] - df["current_liabilities"]) / df["total_assets"]
)

df["retained_earnings_ratio"] = df["retained_earnings"] / df["total_assets"]
df["retained_earnings_to_equity"] = (
    df["retained_earnings"] / df["equity"].replace(0, pd.NA)
)

df["capital_ratio"] = df["capital"] / df["total_assets"]


In [41]:
new_order = [
    "stock_code", "year",

    # 규모
    "total_assets", "capital", "equity", "retained_earnings",

    # 수익성
    "roa", "roe", "net_margin", "op_margin",

    # 성장성
    "sales_growth_rate", "asset_growth",
    "op_income_growth", "net_income_growth",

    # 안정성 / 레버리지 (기존 + 신규)
    "debt_ratio", "debt_to_assets",
    "non_current_debt_ratio", "current_debt_ratio",
    "equity_ratio", "equity_multiplier", "short_term_debt_ratio",

    # 유동성
    "current_ratio",
    "current_assets_ratio", "non_current_assets_ratio",
    "net_working_capital_ratio",

    # 활동성
    "asset_turnover_ratio",

    # 자본구조
    "retained_earnings_ratio",
    "retained_earnings_to_equity",
    "capital_ratio",

    # 구조 변수
    "current_assets", "non_current_assets",
    "current_liabilities", "non_current_liabilities",
    "total_liabilities"
]

df = df.loc[:, new_order]


In [42]:
df.to_sql("financials_v2", conn, if_exists="replace", index=False)
conn.close()

In [43]:
conn = sqlite3.connect(db_path)
pd.read_sql("SELECT * FROM financials_v2 LIMIT 5", conn)


,stock_code,year,total_assets,capital,equity,retained_earnings,roa,roe,net_margin,op_margin,...,net_working_capital_ratio,asset_turnover_ratio,retained_earnings_ratio,retained_earnings_to_equity,capital_ratio,current_assets,non_current_assets,current_liabilities,non_current_liabilities,total_liabilities
0,372800,2021,4.914109e+10,2.888490e+09,2.888490e+09,3.647553e+09,-0.053106,-0.903485,-0.038996,-0.037706,...,0.571858,1.361833,0.074226,1.2627889128922,0.058780,4.230158e+10,6.839509e+09,1.419986e+10,4.844733e+09,1.904459e+10
1,372800,2022,4.167134e+10,2.988495e+09,2.988495e+09,-6.312810e+09,-0.240173,-3.348948,-0.183398,-0.207711,...,0.349614,1.309569,-0.151490,-2.11237079566805,0.071716,3.058691e+10,1.108443e+10,1.601802e+10,5.211660e+08,1.653918e+10
2,372800,2023,4.815975e+10,3.013495e+09,3.013495e+09,-1.306519e+10,-0.147384,-2.355391,-0.094704,-0.085232,...,0.126449,1.556263,-0.271289,-4.33556072766008,0.062573,3.663765e+10,1.152210e+10,3.054788e+10,3.702691e+08,3.091815e+10
3,372800,2024,5.937634e+10,3.013495e+09,3.013495e+09,-1.163791e+10,0.020907,0.411933,0.015599,0.002093,...,0.085053,1.340233,-0.196003,-3.86193154261082,0.050752,4.604766e+10,1.332867e+10,4.099755e+10,2.202691e+08,4.121782e+10
4,372800,2025,4.696405e+10,3.023495e+09,3.023495e+09,-1.743624e+10,-0.070209,-1.090559,-0.160892,0.015322,...,-0.090659,0.436375,-0.371268,-5.76691503706803,0.064379,3.026478e+10,1.669927e+10,3.452251e+10,2.458093e+08,3.476832e+10


In [44]:
from google.colab import files
files.download("종목별 재주재표 분석.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>